In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

In [3]:
load_dotenv()
model = ChatGroq(model="llama-3.3-70b-versatile")

In [4]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):
    prompt = f"Generate a joke on the given topic.\n {state['topic']}"

    joke = model.invoke(prompt)

    return {"joke": joke.content}


def generate_explanation(state: JokeState):
    prompt = f"Generate an explanation on the given joke.\n {state['joke']}"

    explanation = model.invoke(prompt)

    return {"explanation": explanation.content}

In [6]:
graph = StateGraph(JokeState)

graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [7]:
config = {"configurable": {"thread_id": "1"}}

joke = workflow.invoke({"topic": "AI"}, config=config)
print(joke)

{'topic': 'AI', 'joke': 'Why did the AI program go to therapy?\n\nBecause it was struggling to process its emotions.', 'explanation': 'This joke is a play on the concept of artificial intelligence (AI) and its limitations in understanding and managing human-like emotions. The punchline "it was struggling to process its emotions" is a clever wordplay, as AI programs are designed to process and analyze vast amounts of data, but the term "process" in this context is used to refer to the emotional struggles that humans often face.\n\nThe joke relies on the double meaning of the word "process": in computer science, it refers to the act of executing a set of instructions or handling data, but in human psychology, it refers to the act of dealing with and managing emotions. The humor comes from the unexpected twist of applying a technical term to a human-like problem, highlighting the irony that an AI program, designed to be efficient and logical, is seeking therapy to cope with emotional stru

In [8]:
workflow.get_state(config)

StateSnapshot(values={'topic': 'AI', 'joke': 'Why did the AI program go to therapy?\n\nBecause it was struggling to process its emotions.', 'explanation': 'This joke is a play on the concept of artificial intelligence (AI) and its limitations in understanding and managing human-like emotions. The punchline "it was struggling to process its emotions" is a clever wordplay, as AI programs are designed to process and analyze vast amounts of data, but the term "process" in this context is used to refer to the emotional struggles that humans often face.\n\nThe joke relies on the double meaning of the word "process": in computer science, it refers to the act of executing a set of instructions or handling data, but in human psychology, it refers to the act of dealing with and managing emotions. The humor comes from the unexpected twist of applying a technical term to a human-like problem, highlighting the irony that an AI program, designed to be efficient and logical, is seeking therapy to cop

In [9]:
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'AI', 'joke': 'Why did the AI program go to therapy?\n\nBecause it was struggling to process its emotions.', 'explanation': 'This joke is a play on the concept of artificial intelligence (AI) and its limitations in understanding and managing human-like emotions. The punchline "it was struggling to process its emotions" is a clever wordplay, as AI programs are designed to process and analyze vast amounts of data, but the term "process" in this context is used to refer to the emotional struggles that humans often face.\n\nThe joke relies on the double meaning of the word "process": in computer science, it refers to the act of executing a set of instructions or handling data, but in human psychology, it refers to the act of dealing with and managing emotions. The humor comes from the unexpected twist of applying a technical term to a human-like problem, highlighting the irony that an AI program, designed to be efficient and logical, is seeking therapy to co